In [0]:
# ============================================================
# DE-SkyPrint
# 01_ingest_bronze
# Landing -> Bronze
# ============================================================

LANDING = "abfss://landing@skyprint74815.dfs.core.windows.net/"
BRONZE  = "abfss://bronze@skyprint74815.dfs.core.windows.net/"

SOURCES = ["opensky", "openmeteo"]


# ------------------------------------------------------------
# 1. Get all files recursively
# ------------------------------------------------------------
def list_files_recursive(path):
    files = []

    try:
        items = dbutils.fs.ls(path)
    except Exception:
        return files

    for item in items:
        if item.path.endswith("/"):
            files.extend(list_files_recursive(item.path))
        else:
            files.append(item)

    return files


# ------------------------------------------------------------
# 2. Check if a file exists and return its size
# ------------------------------------------------------------
def get_file_size(path):
    parent = path.rsplit("/", 1)[0] + "/"
    filename = path.rsplit("/", 1)[1]

    try:
        for item in dbutils.fs.ls(parent):
            if item.name.rstrip("/") == filename:
                return item.size
    except Exception:
        pass

    return None


# ------------------------------------------------------------
# 3. Process Landing files
# ------------------------------------------------------------
total_found = 0
total_copied = 0
total_removed = 0
total_already_exists = 0
errors = []


for source in SOURCES:

    landing_source = f"{LANDING}{source}/"

    print("\n" + "=" * 60)
    print(f"Processing source: {source}")
    print("=" * 60)

    files = list_files_recursive(landing_source)

    print(f"Files found in Landing: {len(files)}")

    total_found += len(files)

    for file_info in files:

        src = file_info.path

        # Keep the exact same relative path
        # Example:
        # landing/opensky/year=2026/month=09/day=27/file.json
        # becomes:
        # bronze/opensky/year=2026/month=09/day=27/file.json

        relative_path = src[len(LANDING):]
        dst = BRONZE + relative_path

        source_size = file_info.size

        try:

            # ------------------------------------------------
            # A. Check whether destination already exists
            # ------------------------------------------------
            destination_size = get_file_size(dst)

            if destination_size is not None:

                if destination_size == source_size:
                    print(f"Already in Bronze: {relative_path}")

                    # Same file size -> remove duplicate from Landing
                    deleted = dbutils.fs.rm(src)

                    if not deleted:
                        raise Exception(
                            "File exists in Bronze but could not "
                            "be removed from Landing"
                        )

                    total_already_exists += 1
                    total_removed += 1
                    continue

                else:
                    # IMPORTANT:
                    # Do NOT overwrite an existing Bronze file
                    # if the sizes are different.
                    raise Exception(
                        f"Destination already exists with different size. "
                        f"Landing={source_size}, Bronze={destination_size}"
                    )


            # ------------------------------------------------
            # B. Create destination directory
            # ------------------------------------------------
            destination_parent = dst.rsplit("/", 1)[0] + "/"

            dbutils.fs.mkdirs(destination_parent)


            # ------------------------------------------------
            # C. Copy Landing -> Bronze
            # ------------------------------------------------
            copied = dbutils.fs.cp(src, dst)

            if not copied:
                raise Exception("Copy to Bronze returned False")


            # ------------------------------------------------
            # D. Verify copied file
            # ------------------------------------------------
            copied_size = get_file_size(dst)

            if copied_size is None:
                raise Exception(
                    "File was copied but destination could not be verified"
                )

            if copied_size != source_size:
                raise Exception(
                    f"File size verification failed. "
                    f"Landing={source_size}, Bronze={copied_size}"
                )


            # ------------------------------------------------
            # E. Delete from Landing ONLY after verification
            # ------------------------------------------------
            deleted = dbutils.fs.rm(src)

            if not deleted:
                raise Exception(
                    "Copy verified, but source could not be deleted "
                    "from Landing"
                )


            total_copied += 1
            total_removed += 1

            print(f"✓ Ingested: {relative_path}")


        except Exception as e:

            errors.append(
                {
                    "file": relative_path,
                    "error": str(e)
                }
            )

            print(f"✗ FAILED: {relative_path}")
            print(f"  {e}")


# ------------------------------------------------------------
# 4. Final summary
# ------------------------------------------------------------
print("\n")
print("=" * 60)
print("INGESTION SUMMARY")
print("=" * 60)

print(f"Files found          : {total_found}")
print(f"New files copied     : {total_copied}")
print(f"Already in Bronze    : {total_already_exists}")
print(f"Removed from Landing : {total_removed}")
print(f"Errors               : {len(errors)}")


# ------------------------------------------------------------
# 5. Fail the Databricks task if anything went wrong
# ------------------------------------------------------------
if errors:

    print("\nFAILED FILES:")

    for error in errors:
        print(f"- {error['file']}")
        print(f"  {error['error']}")

    raise Exception(
        f"Ingestion failed for {len(errors)} file(s). "
        "Pipeline stopped to protect the data."
    )


print("\n✓ Landing -> Bronze ingestion completed successfully")


Processing source: opensky
Files found in Landing: 0

Processing source: openmeteo
Files found in Landing: 50
✓ Ingested: openmeteo/year=2026/month=09/day=23/openmeteo_2026-09-23_16-14-59.json
✓ Ingested: openmeteo/year=2026/month=09/day=23/openmeteo_2026-09-23_17-29-59.json
✓ Ingested: openmeteo/year=2026/month=09/day=23/openmeteo_2026-09-23_17-45-00.json
✓ Ingested: openmeteo/year=2026/month=09/day=23/openmeteo_2026-09-23_18-00-00.json
✓ Ingested: openmeteo/year=2026/month=09/day=23/openmeteo_2026-09-23_20-44-59.json
✓ Ingested: openmeteo/year=2026/month=09/day=23/openmeteo_2026-09-23_23-44-59.json
✓ Ingested: openmeteo/year=2026/month=09/day=24/openmeteo_2026-09-24_01-59-59.json
✓ Ingested: openmeteo/year=2026/month=09/day=24/openmeteo_2026-09-24_02-14-59.json
✓ Ingested: openmeteo/year=2026/month=09/day=24/openmeteo_2026-09-24_03-45-00.json
✓ Ingested: openmeteo/year=2026/month=09/day=24/openmeteo_2026-09-24_04-15-00.json
✓ Ingested: openmeteo/year=2026/month=09/day=24/openmeteo_2